# tsfm-scale-identifiability: Scale-Ratio Non-Identifiability in Universal TSFMs
**Repository:** [`jaimejarauta/decoupled-delta-net`](https://github.com/jaimejarauta/decoupled-delta-net)
**Branch:** `feature/tsfm-scale-identifiability`
**Lead Researcher:** Jaime Jarauta
**One-Click Run:** Select **Runtime -> Run all**.


In [ ]:
# Step 1: Self-Healing Environment Setup & Bulletproof Credit Guard
import os, sys, signal, subprocess
from IPython import get_ipython

def auto_teardown(signum=None, frame=None):
    print("\n[Credit Guard] Unhandled error, interrupt, or completion! Syncing W&B and unassigning GPU...")
    try:
        subprocess.run("wandb sync --include-offline 2>/dev/null || true", shell=True)
    except Exception:
        pass
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception:
        pass

def custom_exc_handler(self, etype, value, tb, tb_offset=None):
    self.showtraceback((etype, value, tb), tb_offset=tb_offset)
    auto_teardown()
    return []

ip = get_ipython()
if ip is not None:
    ip.set_custom_exc((Exception,), custom_exc_handler)

print("[Ready] Credit guard active. VM will self-terminate on error or completion.")


In [ ]:
# Step 2: Clone Repository (feature/tsfm-scale-identifiability) & Mount Google Drive
import os, sys, subprocess
from pathlib import Path
import torch

assert torch.cuda.is_available(), "Switch Colab runtime to GPU: Runtime -> Change runtime type -> GPU"
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"GPU: {gpu_name} | VRAM: {vram_gb:.1f} GB | BF16: {torch.cuda.is_bf16_supported()}")

repo_url = "https://github.com/jaimejarauta/decoupled-delta-net.git"
branch = "feature/tsfm-scale-identifiability"
target_dir = "/content/tsfm-scale-identifiability"

%cd /content
if not os.path.exists(target_dir):
    print(f"Cloning feature/tsfm-scale-identifiability from {repo_url}...")
    subprocess.run(["git", "clone", "-b", branch, repo_url, target_dir], check=True)
else:
    print(f"Updating feature/tsfm-scale-identifiability to latest...")
    subprocess.run(["git", "-C", target_dir, "fetch", "origin", branch], check=True)
    subprocess.run(["git", "-C", target_dir, "checkout", branch], check=True)
    subprocess.run(["git", "-C", target_dir, "reset", "--hard", f"origin/{branch}"], check=True)
    subprocess.run(["git", "-C", target_dir, "pull", "origin", branch], check=True)

%cd /content/tsfm-scale-identifiability
!pip install -q -e .
!pip install -q "chronos-forecasting>=2.0.0" "fev>=0.6.0" "lightgbm>=4.0.0" "datasets>=3.0.0" "pandas[pyarrow]" accelerate pytest wandb

persist_dir = Path("/content/tsfm-scale-identifiability/results")
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    persist_dir = Path("/content/drive/MyDrive/moonshot_results/tsfm-scale-identifiability")
    persist_dir.mkdir(parents=True, exist_ok=True)
    print(f"[Google Drive] Persistent results directory mounted: {persist_dir}")
except Exception as e:
    print(f"Drive notice: {e}")
    persist_dir.mkdir(parents=True, exist_ok=True)

if "/content/tsfm-scale-identifiability/src" not in sys.path:
    sys.path.insert(0, "/content/tsfm-scale-identifiability/src")
os.environ["PYTHONPATH"] = "/content/tsfm-scale-identifiability/src:" + os.environ.get("PYTHONPATH", "")
print("[Ready] tsfm-scale-identifiability environment ready on GPU.")


In [ ]:
# Step 3: Fast CPU Unit Verification Tests
!pytest -q tests/test_identifiability.py


In [ ]:
# Step 4: Fast Smoke Pilot (Live Streaming)
print("=" * 70)
print("STEP 4: FAST SMOKE PILOT")
print("=" * 70)
!python scripts/run_kill.py run --model amazon/chronos-2 --max-tasks 3 --max-series-per-window 64 --out {persist_dir}/smoke_pilot.jsonl
!python scripts/run_kill.py report --out {persist_dir}/smoke_pilot.jsonl


In [ ]:
# Step 5: Full Resumable Pre-Registered Kill Sweep (Live Streaming)
print("=" * 70)
print("STEP 5: FULL RESUMABLE PRE-REGISTERED KILL SWEEP")
print("=" * 70)
!python scripts/run_kill.py run --model amazon/chronos-2 --max-tasks 46 --max-series-per-window 128 --out {persist_dir}/kill.jsonl
!python scripts/run_kill.py report --out {persist_dir}/kill.jsonl


In [ ]:
# Step 6: Sync Telemetry & Unassign GPU (Zero Credit Leakage)
print("[Credit Guard] Sweep complete! Syncing W&B and unassigning GPU...")
auto_teardown()
